# Stock and COE features: one row per rent-approval month

Builds the two monthly temporal feature tables that join to the rental data on `RENT_APPROVAL_YEAR` + `RENT_APPROVAL_MONTH`.

| | |
|---|---|
| **Inputs** | `data/original_dataset/auxiliary/sg-stock-prices.csv` (daily prices of 77 stocks), `data/original_dataset/auxiliary/sg-coe-prices.csv` (COE bidding rounds) |
| **Join check** | `data/original_dataset/train.csv`, `test.csv` (every row must find its month) |
| **Outputs** | `data/features/cleaned/temporal_features/stock_features.csv`, `coe_features_monthly.csv` |

In [1]:
import io
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)


def find_project_root(start=None):
    """Walk up from the working directory to the folder that holds data/config.py (final_report)."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / 'data' / 'config.py').exists():
            return folder
    raise FileNotFoundError(f'no data/config.py found above {start}')


ROOT = find_project_root()
ORIGINAL = ROOT / 'data' / 'original_dataset'
AUXILIARY = ORIGINAL / 'auxiliary'
OUTPUT = ROOT / 'data' / 'features' / 'cleaned' / 'temporal_features'
OUTPUT.mkdir(parents=True, exist_ok=True)

KEY = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH']          # the join grain of both tables
print('project root:', ROOT)

project root: /Users/keltonn/Desktop/Projects/housing_rent_price/final_report


## 1. Stock features

One row per month, three columns per stock: `stk_<name>_log_price_mean`, `stk_<name>_mom_return`, `stk_<name>_return_volatility`.

Everything is computed on the log of the **adjusted close** (it accounts for dividends and splits), so daily changes are log returns and moves are comparable between cheap and
expensive stocks.

In [2]:
prices = pd.read_csv(AUXILIARY / 'sg-stock-prices.csv')

# rows of non-trading days have no prices at all
stock = prices.dropna(subset=['ADJUSTED_CLOSE']).copy()
print(f'{len(prices):,} rows, {len(prices) - len(stock)} without prices dropped; {stock["NAME"].nunique()} stocks, {stock["DATE"].min()} to {stock["DATE"].max()}')

91,642 rows, 85 without prices dropped; 77 stocks, 2021-01-04 to 2026-07-30


### 1.1 Monthly level and volatility

The month's average log price (its level) and the standard deviation of the daily log returns within the month (its volatility).

In [3]:
def monthly_level_and_volatility(stock):
    stock = stock.copy()
    stock['DATE'] = pd.to_datetime(stock['DATE'])
    stock['RENT_APPROVAL_YEAR'] = stock['DATE'].dt.year
    stock['RENT_APPROVAL_MONTH'] = stock['DATE'].dt.month
    stock = stock.sort_values(['NAME', 'DATE'])

    stock['LOG_PRICE'] = np.log(stock['ADJUSTED_CLOSE'])
    stock['DAILY_LOG_RETURN'] = stock.groupby('NAME')['LOG_PRICE'].diff()

    # one row per stock x year+month
    return (stock.groupby(['NAME'] + KEY)
            .agg(LOG_PRICE_MEAN=('LOG_PRICE', 'mean'),
                 RETURN_VOLATILITY=('DAILY_LOG_RETURN', 'std'))
            .reset_index())


monthly_stock = monthly_level_and_volatility(stock)
monthly_stock.head()

,NAME,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LOG_PRICE_MEAN,RETURN_VOLATILITY
0,Antalpha Platform,2025,5,2.519028,0.135667
1,Antalpha Platform,2025,6,2.486008,0.056202
2,Antalpha Platform,2025,7,2.461962,0.039747
3,Antalpha Platform,2025,8,2.512603,0.031019
4,Antalpha Platform,2025,9,2.499707,0.025648


### 1.2 Month-over-month return

The change in the monthly *average* log price versus the previous month, so it reflects the month's overall conditions rather than where the last trading day happened to close.
It is computed on a complete month index so a missing month can never be silently treated as the "previous" month. The first month of each stock is NaN.

In [4]:
def add_month_over_month_return(monthly):
    monthly = monthly.copy()
    monthly['MONTH'] = pd.PeriodIndex(year=monthly['RENT_APPROVAL_YEAR'], month=monthly['RENT_APPROVAL_MONTH'], freq='M')
    mean_price = monthly.pivot(index='MONTH', columns='NAME', values='LOG_PRICE_MEAN')
    mean_price = mean_price.reindex(pd.period_range(mean_price.index.min(), mean_price.index.max(), freq='M'))
    mom = (mean_price.diff()
           .stack(dropna=False)
           .rename('MOM_RETURN')
           .rename_axis(['MONTH', 'NAME'])
           .reset_index())
    return monthly.merge(mom, on=['MONTH', 'NAME'], how='left').drop(columns=['MONTH'])


monthly_stock = add_month_over_month_return(monthly_stock)
monthly_stock.head()

,NAME,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LOG_PRICE_MEAN,RETURN_VOLATILITY,MOM_RETURN
0,Antalpha Platform,2025,5,2.519028,0.135667,NaN
1,Antalpha Platform,2025,6,2.486008,0.056202,-0.033020
2,Antalpha Platform,2025,7,2.461962,0.039747,-0.024046
3,Antalpha Platform,2025,8,2.512603,0.031019,0.050641
4,Antalpha Platform,2025,9,2.499707,0.025648,-0.012897


### 1.3 One row per month

Pivot to one row per year+month, one column per stock x feature. Stocks that were not yet listed in a month are left as NaN (not 0) so the model can tell them apart.

In [5]:
def stock_feature_table(monthly):
    features = monthly.melt(id_vars=['NAME'] + KEY, var_name='FEATURE')
    features['COL'] = ('stk_'
                       + features['NAME'].str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
                       + '_'
                       + features['FEATURE'].str.lower())
    table = features.pivot(index=KEY, columns='COL', values='value').reset_index()
    table.columns.name = None
    return table


stock_features = stock_feature_table(monthly_stock)
print('stock features:', stock_features.shape, '(months x [2 key columns + 3 per stock])')
stock_features.head()

stock features: (67, 233) (months x [2 key columns + 3 per stock])


,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,stk_antalpha_platform_log_price_mean,stk_antalpha_platform_mom_return,stk_antalpha_platform_return_volatility,stk_ascendas_reit_log_price_mean,stk_ascendas_reit_mom_return,stk_ascendas_reit_return_volatility,stk_belive_holdings_log_price_mean,stk_belive_holdings_mom_return,...,stk_wave_life_sciences_return_volatility,stk_webuy_global_log_price_mean,stk_webuy_global_mom_return,stk_webuy_global_return_volatility,stk_wilmar_international_log_price_mean,stk_wilmar_international_mom_return,stk_wilmar_international_return_volatility,stk_yanlord_log_price_mean,stk_yanlord_mom_return,stk_yanlord_return_volatility
0,2021,1,NaN,NaN,NaN,0.806770,NaN,0.012960,NaN,NaN,...,0.043841,NaN,NaN,NaN,1.402303,NaN,0.014903,0.001021,NaN,0.012827
1,2021,2,NaN,NaN,NaN,0.810418,0.003648,0.010391,NaN,NaN,...,0.043720,NaN,NaN,NaN,1.440095,0.037792,0.012453,-0.009466,-0.010488,0.007062
2,2021,3,NaN,NaN,NaN,0.786484,-0.023934,0.012611,NaN,NaN,...,0.075097,NaN,NaN,NaN,1.427988,-0.012107,0.008999,0.039096,0.048563,0.011887
3,2021,4,NaN,NaN,NaN,0.824610,0.038125,0.006253,NaN,NaN,...,0.027922,NaN,NaN,NaN,1.446436,0.018448,0.007545,0.133329,0.094233,0.011101
4,2021,5,NaN,NaN,NaN,0.792135,-0.032474,0.012063,NaN,NaN,...,0.046543,NaN,NaN,NaN,1.363611,-0.082825,0.015108,0.180473,0.047144,0.019049


## 2. COE features

Each row of the raw file is one COE bidding round (two per month, a few months have one). Four categories are in the file: **A** (cars up to 1600cc), **B** (cars above 1600cc),
**C** (goods vehicles and buses) and **E** (open category). There is no D (motorcycle) category.

- `PREVAILING_QUOTA_PREMIUM` is **not used**: it changes with the age of the licence (like the yield of a bond part-way to maturity), so it is not a market price at the time of the bid.
- The three features are `QUOTA_PREMIUM` (the price, $), `QUOTA` (licences on offer) and `BIDS_RECEIVED`. Money and thousands separators are removed. The monthly value of a
  feature is the **mean over that month's rounds**, so months with one round are comparable with months with two.
- The result has one row per year-month and one column per category x feature (`coe_<category>_<feature>`), keyed like `stock_features.csv`.

In [6]:
coe = pd.read_csv(AUXILIARY / 'sg-coe-prices.csv')

MONTHS = {m: i + 1 for i, m in enumerate(['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])}
COE_FEATURES = ['QUOTA_PREMIUM', 'QUOTA', 'BIDS_RECEIVED']


def clean_coe_rounds(coe):
    clean = coe.drop(columns='PREVAILING_QUOTA_PREMIUM').copy()
    for col in COE_FEATURES:
        clean[col] = clean[col].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)
    clean['RENT_APPROVAL_YEAR'] = clean['YEAR']
    clean['RENT_APPROVAL_MONTH'] = clean['MONTH'].map(MONTHS)
    assert clean['RENT_APPROVAL_MONTH'].notna().all() and (clean[COE_FEATURES] > 0).all().all()
    return clean


coe_rounds = clean_coe_rounds(coe)

rounds = coe_rounds.groupby(['CATEGORY'] + KEY).size()
print('rounds per category-month:', rounds.value_counts().to_dict(), '| months with one round:')
print(rounds[rounds == 1].reset_index().rename(columns={0: 'rounds'}).to_string(index=False))

rounds per category-month: {2: 268, 1: 3} | months with one round:
CATEGORY  RENT_APPROVAL_YEAR  RENT_APPROVAL_MONTH  rounds
       a                2026                    8       1
       b                2026                    8       1
       c                2026                    8       1


In [7]:
def coe_feature_table(rounds):
    monthly = rounds.groupby(KEY + ['CATEGORY'])[COE_FEATURES].mean().unstack('CATEGORY')
    monthly.columns = [f'coe_{category}_{feature.lower()}' for feature, category in monthly.columns]
    return monthly[sorted(monthly.columns)].reset_index()


coe_features = coe_feature_table(coe_rounds)
first, last = coe_features.iloc[0], coe_features.iloc[-1]
print(f'{len(coe_features)} months, {int(first.RENT_APPROVAL_YEAR)}-{int(first.RENT_APPROVAL_MONTH):02d} to {int(last.RENT_APPROVAL_YEAR)}-{int(last.RENT_APPROVAL_MONTH):02d}; '
      f'{coe_features.shape[1] - 2} feature columns; missing values: {int(coe_features.isna().sum().sum())}')
coe_features.round(1).tail(8)

68 months, 2021-01 to 2026-08; 12 feature columns; missing values: 3


,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,coe_a_bids_received,coe_a_quota,coe_a_quota_premium,coe_b_bids_received,coe_b_quota,coe_b_quota_premium,coe_c_bids_received,coe_c_quota,coe_c_quota_premium,coe_e_bids_received,coe_e_quota,coe_e_quota_premium
60,2026,1,1980.5,1293.0,105755.0,1167.5,813.5,120367.0,414.0,285.0,75352.5,477.0,288.5,121445.5
61,2026,2,1978.5,1277.0,106410.5,1159.5,813.0,107945.5,461.0,293.0,74900.0,516.5,255.0,114445.0
62,2026,3,1828.5,1266.0,110055.0,1223.0,813.5,114785.0,463.0,290.0,77000.0,468.0,245.5,116504.5
63,2026,4,2473.5,1265.0,120505.0,1300.0,811.0,121000.5,516.5,292.5,81751.0,484.5,243.0,123001.5
64,2026,5,2177.0,1270.0,124509.5,1400.5,876.0,127868.5,489.5,292.5,89851.0,457.5,255.0,128850.0
65,2026,6,1922.0,1248.5,124928.0,1251.0,886.0,125245.5,421.5,292.0,93500.5,492.0,261.5,129001.0
66,2026,7,1734.5,1244.5,127500.0,1432.5,869.0,130389.5,436.0,298.5,94444.5,451.0,270.0,129886.0
67,2026,8,1522.0,1226.0,123890.0,1316.0,926.0,129910.0,465.0,315.0,91545.0,NaN,NaN,NaN


## 3. Check the join against the rental data

Every train and test row must find its month in both tables (`many_to_one`). Only the stock features may have gaps: a stock that was not yet listed has NaN for its early months,
and no stock has a return in its first month.

In [8]:
def rent_months(name):
    """Year and month of the rows of data/original_dataset/<name>.csv (RENT_APPROVAL_DATE is 'YYYY-MM')."""
    date = pd.read_csv(ORIGINAL / f'{name}.csv', usecols=['RENT_APPROVAL_DATE'])['RENT_APPROVAL_DATE']
    return pd.DataFrame({'RENT_APPROVAL_YEAR': date.str[:4].astype(int), 'RENT_APPROVAL_MONTH': date.str[5:7].astype(int)})


for name in ('train', 'test'):
    rows = rent_months(name)
    for table_name, table, probe in [('stock', stock_features, 'stk_dbs_group_log_price_mean'), ('COE', coe_features, 'coe_a_quota_premium')]:
        joined = rows.merge(table, on=KEY, how='left', validate='many_to_one')
        assert len(joined) == len(rows)
        print(f'{name:5s} {table_name:5s}: {len(rows):,} rows, {rows.drop_duplicates().shape[0]} months, rows without features: {int(joined[probe].isna().sum())}')
        assert joined[probe].isna().sum() == 0

train stock: 150,000 rows, 51 months, rows without features: 0
train COE  : 150,000 rows, 51 months, rows without features: 0
test  stock: 50,000 rows, 17 months, rows without features: 0
test  COE  : 50,000 rows, 17 months, rows without features: 0


## 4. Save

The two tables are written to `data/features/cleaned/temporal_features/`, where the modelling pipeline reads them. If a file is already there, the new table is first compared with it
(the COE table is written to 2 decimals, so it is compared in that form), so a rerun shows whether the previous features are reproduced.

In [9]:
def save(table, path, **to_csv_kwargs):
    text = table.to_csv(index=False, **to_csv_kwargs)
    if path.exists():
        try:
            pd.testing.assert_frame_equal(pd.read_csv(io.StringIO(text)), pd.read_csv(path), rtol=1e-9)
            print(f'{path.name}: identical to the existing file')
        except AssertionError as error:
            print(f'{path.name}: DIFFERS from the existing file ({str(error).splitlines()[0]})')
    path.write_text(text)
    print(f'{path.name}: saved, {table.shape[0]} rows x {table.shape[1]} columns')


save(stock_features, OUTPUT / 'stock_features.csv')
save(coe_features, OUTPUT / 'coe_features_monthly.csv', float_format='%.2f')

stock_features.csv: identical to the existing file
stock_features.csv: saved, 67 rows x 233 columns
coe_features_monthly.csv: identical to the existing file
coe_features_monthly.csv: saved, 68 rows x 14 columns
